# 06b · Batch-level permutation test with tree-embedding statistics

**Why this notebook exists.** Notebooks 04 and 06 on the real Adult data showed:
- the per-row, conformally valid tree-embedding score (`ref_aphd`) has almost no power at alpha = 0.05 (TPR 0.001-0.022) and Isolation Forest dominates every per-row valid detector we have;
- TOOD's batch mode was the strongest score at high contamination (AUROC up to 1.0) but has **no guarantee**.

A batch-level **permutation test** keeps TOOD's strength and gives an **exactly valid** p-value: under H0 ("the batch is iid from the ID distribution") the pooled reference + batch rows are exchangeable, whatever the statistic does with the rest of the batch.

Four statistics, all computed on the **same** permutations:
- `tood`: within-batch leaf collision (TOOD's own quantity), two-sided;
- `rf_mmd`: unbiased MMD² between batch and reference with the random-forest kernel, upper tail;
- `mean_iforest`, `mean_msp`: batch mean of the per-row score (the obvious baselines), two-sided.

**Scope and honesty.** This tests the batch, not individual rows. A kernel two-sample statistic with a forest kernel is a standard construction; before claiming anything it must be compared with classifier two-sample tests (C2ST) and with MMD using standard kernels. Do the literature check first.

In [1]:
import numpy as np
import pandas as pd

from conformal_trees.conformal import batch_permutation_test
from conformal_trees.data import build_covariate_shifts, load_dataset, make_splits, mix_batch
from conformal_trees.models import GBDTClassifier, TreeEmbedder
from conformal_trees.signals import IsolationForestScore, LeafIndex, msp_score
from conformal_trees.utils import derive_seed, load_config, make_rng

pd.set_option("display.max_columns", 50)
cfg = load_config("base.yaml", "models.yaml", "ood_scenarios.yaml")
ALPHA = cfg["conformal"]["alpha"]
SEEDS = cfg["project"]["seeds"][:3]       # use SEEDS[:1] for a quick look
M = 1000                                  # batch size
N_PERM = 199                              # permutations per test
N_H0 = 60                                 # clean batches per seed (validity)
N_POWER = 15                              # contaminated batches per seed and configuration
PIS = (0.05, 0.1, 0.25, 0.5)              # contamination levels
SCENARIOS = ("mean_shift@1", "gaussian_noise@2")
STATS = ["tood", "rf_mmd", "mean_iforest", "mean_msp"]
ds = load_dataset("adult")

## 1. Run the protocol (the slow cell)
Per seed: fit on `train`, use `calib` as the clean reference, draw batches of `M` rows from `test`.
`pi = 0` batches are clean (validity). For `pi > 0` a fraction of the rows is replaced by their shifted copies (`mix_batch`).

In [4]:
import numpy as np
import pandas as pd

def draw_batch(X_test, shift, pi, seed, tag, rep):
    idx = make_rng(derive_seed(seed, "batch", tag, rep)).choice(len(X_test), size=M, replace=False)
    X_b = X_test.iloc[idx].reset_index(drop=True)
    if pi == 0.0:
        return X_b
    X_ood = shift.apply(X_b, seed=derive_seed(seed, "shift", tag, rep))
    return mix_batch(X_b, X_ood, pi, seed=derive_seed(seed, "mix", tag, rep))[0]


records = []
for seed in SEEDS:
    sp = make_splits(ds.X, ds.y, cfg["splits"], seed=seed)
    emb = TreeEmbedder.from_config(cfg["embedder"], ds.numeric, ds.categorical, seed=seed).fit(sp.train.X, sp.train.y)
    gbdt = GBDTClassifier.from_config(cfg["gbdt"], seed=seed).fit(sp.train.X, sp.train.y)
    iforest = IsolationForestScore(ds.numeric, ds.categorical, seed=seed).fit(sp.train.X)
    
    li = LeafIndex.from_embedder(emb)
    
    G_cal = li.to_global(emb.transform(sp.calib.X))
    cal_if = iforest.score(sp.calib.X)
    cal_msp = msp_score(gbdt.predict_proba(sp.calib.X)[:, 1])
    
    shifts = {s.name: s for s in build_covariate_shifts(cfg, sp.train.X, ds.numeric) if s.name in SCENARIOS}

    def run(batch, rng):
        G_batch = li.to_global(emb.transform(batch))
        G = np.vstack([G_cal, G_batch])
        
        scores = {
            "iforest": np.concatenate([cal_if, iforest.score(batch)]),
            "msp": np.concatenate([cal_msp, msp_score(gbdt.predict_proba(batch)[:, 1])]),
        }
        return batch_permutation_test(G, len(batch), li, rng=rng, n_perm=N_PERM, scores=scores)

    # H0 evaluation
    print(f"[Seed {seed}] Evaluating H0 ({N_H0} reps)...")
    for rep in range(N_H0):
        b = draw_batch(sp.test.X, None, 0.0, seed, "h0", rep)
        p = run(b, make_rng(derive_seed(seed, "perm", "h0", rep)))
        records.append({"seed": seed, "scenario": "none", "pi": 0.0, **p})

    # Power evaluation under shifts
    for name, shift in shifts.items():
        print(f"  [Seed {seed}] Running scenario: {name}")
        for pi in PIS:
            tag = f"{name}|{pi}"
            for rep in range(N_POWER):
                b = draw_batch(sp.test.X, shift, pi, seed, tag, rep)
                p = run(b, make_rng(derive_seed(seed, "perm", tag, rep)))
                records.append({"seed": seed, "scenario": name, "pi": pi, **p})

    print(f"--> Seed {seed} done successfully!\n")

res = pd.DataFrame(records)
rejected = res.copy()
rejected[STATS] = res[STATS] <= ALPHA

print("Evaluation finished!")
display(res.head())

d:\Development\projects for master\Paper researcher\Conformalized Tree-Based Gradient Ensembles for Out-of-Distribution Detection and Recourse\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[Seed 0] Evaluating H0 (60 reps)...
  [Seed 0] Running scenario: mean_shift@1
  [Seed 0] Running scenario: gaussian_noise@2
--> Seed 0 done successfully!



d:\Development\projects for master\Paper researcher\Conformalized Tree-Based Gradient Ensembles for Out-of-Distribution Detection and Recourse\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[Seed 1] Evaluating H0 (60 reps)...
  [Seed 1] Running scenario: mean_shift@1
  [Seed 1] Running scenario: gaussian_noise@2
--> Seed 1 done successfully!



d:\Development\projects for master\Paper researcher\Conformalized Tree-Based Gradient Ensembles for Out-of-Distribution Detection and Recourse\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[Seed 2] Evaluating H0 (60 reps)...
  [Seed 2] Running scenario: mean_shift@1
  [Seed 2] Running scenario: gaussian_noise@2
--> Seed 2 done successfully!

Evaluation finished!


,seed,scenario,pi,tood,rf_mmd,mean_iforest,mean_msp
0,0,none,0.0,0.13,0.115,0.66,0.97
1,0,none,0.0,0.03,0.040,0.12,0.38
2,0,none,0.0,0.28,0.110,0.22,0.50
3,0,none,0.0,0.79,0.215,0.32,0.07
4,0,none,0.0,0.15,0.315,0.36,0.34


## 2. Validity under H0 (clean batches)
Rejection rate at alpha. The test is exactly valid, so every statistic should sit at alpha up to Monte Carlo noise (`se`).
A value several `se` above alpha means a bug (for example the reference and the batch are not independent, or the scorers saw the pool).

In [5]:
h0 = rejected[rejected.pi == 0.0]
n = len(h0)
print(f"alpha = {ALPHA}, clean batches = {n}, binomial se = {np.sqrt(ALPHA * (1 - ALPHA) / n):.4f}")
h0[STATS].mean().round(4).to_frame("rejection rate")

alpha = 0.05, clean batches = 180, binomial se = 0.0162


,rejection rate
tood,0.0778
rf_mmd,0.0611
mean_iforest,0.0444
mean_msp,0.0389


## 3. Power: rejection rate by scenario and contamination
Fraction of contaminated batches the test flags at alpha (batch size = `M`). Compare the four statistics on the same permutations.

In [6]:
power = rejected[rejected.pi > 0].groupby(["scenario", "pi"])[STATS].mean().round(3)
power

tood  rf_mmd  mean_iforest  mean_msp
scenario         pi                                         
gaussian_noise@2 0.05  0.111   0.133         0.311     0.044
                 0.10  0.089   0.289         0.844     0.067
                 0.25  0.089   1.000         1.000     0.156
                 0.50  0.578   1.000         1.000     0.667
mean_shift@1     0.05  0.267   0.200         0.111     0.111
                 0.10  0.356   0.400         0.289     0.311
                 0.25  0.778   1.000         0.956     0.978
                 0.50  0.467   1.000         1.000     1.000

## 4. Notes to carry into `docs/decisions.md`
- Are all four H0 rejection rates within about 2 se of alpha?
- Which statistic wins at low contamination (`pi` 0.05-0.1) and which at high (`pi` 0.5)? Does `rf_mmd` beat `tood` as it did on the synthetic data?
- Where does `rf_mmd` beat the batch means of Isolation Forest / MSP, and where not? This is the comparison the batch-level claim rests on.
- Not yet done and required before any claim: C2ST and standard-kernel MMD baselines; other batch sizes `M`; a literature check of forest-kernel two-sample tests.